# Aufgabe 1: Log preprocessing

In [10]:
import pandas
import numpy as np

In [7]:
raw_log = pandas.read_csv("customer_request_system_log.csv", parse_dates=["timestamp_start", "timestamp_end"])

raw_log[0:3]

,record_id,customer_id,transaction_id,action_type,status_code,user_id,timestamp_start,timestamp_end,priority_level,response_time_seconds,notes
0,1001,CUST-001,REQ-001,CREATE_ENTRY,200,USR-A,2025-01-01 09:00:00,2025-01-01 09:01:30,HIGH,90,Initial entry for customer inquiry
1,1002,CUST-002,REQ-002,DATA_INPUT,200,USR-B,2025-01-01 09:15:00,2025-01-01 09:20:00,MEDIUM,300,Customer details entered
2,1003,CUST-001,REQ-001,VALIDATE_DATA,200,USR-C,2025-01-01 09:25:00,2025-01-01 09:27:00,HIGH,120,Data validated and passed


## Actions

**Aktivität**
Eine Aktivität besteht aus "action_type" und "status_code". Daraus ergeben sich 7 actions von denen eine fehlschlagen kann, was insgesamt 8 Aktivitäten macht.
Die anderen Spalten stellen keine Aktivität dar, sondern spezifizieren Details einer Aktivität.
Eg:
 * Die gleich Aktivität kann für verschiedene User durchgeführt werden
 * Die gleiche Aktivität kann unterschiedliche Details (Notes) haben

**CaseID**
Die CaseId ist die Numer in der CustomerID, die sich entweder in der customer oder Transaction spalte befindet

**TimeStamp**
```timestamp_start``` ist die Timestamp. Es ist üblich die startzeit als Timestamp anzugeben. Die Endzeit kann nicht zusätzlich verwendet werden, weil ein Zeitpunkt und keine Zeitspanne gefragt ist.

### Gruppieren der Aktionen
Kombinieren von action und status

In [9]:
# group into list of row dicts, with transformed key
groups = [
    (f"{action}_{'SUCCESS' if status == 200 else 'FAILED'}", grp.to_dict('records'))
    for (action, status), grp in raw_log.groupby(['action_type', 'status_code'])
]

# equivalent of actionGroups.Select(x => x.Item1)
[k for k, _ in groups]


['CHECK_RULES_SUCCESS',
 'CLOSE_CASE_SUCCESS',
 'CREATE_ENTRY_SUCCESS',
 'DATA_INPUT_SUCCESS',
 'DECIDE_ACTION_SUCCESS',
 'REVISE_ENTRY_SUCCESS',
 'VALIDATE_DATA_SUCCESS',
 'VALIDATE_DATA_FAILED']

### Erzeugen der Xes Daten

In [15]:
df = raw_log.copy()

# compute CaseId (vectorized)
case_id_src = df['customer_id'].where(raw_log['customer_id'].str.startswith('CUST-'),
                                      raw_log['transaction_id'])

df['CaseId'] = case_id_src.str[5:].astype(int)

# compute Activity
df['Activity'] = df['action_type'] + '_' + np.where(df['status_code'] == 200, 'SUCCESS', 'FAILED')

# common projection (like new XesGroup(CaseId, Activity, Timestamp))
proj = df[['CaseId', 'Activity', 'timestamp_start']].rename(columns={'timestamp_start': 'Timestamp'})

# xes: order by Timestamp then CaseId
xes = proj.sort_values(['Timestamp', 'CaseId']).reset_index(drop=True)

# xes2: order by CaseId then Timestamp
xes2 = proj.sort_values(['CaseId', 'Timestamp']).reset_index(drop=True)

xes[0:3]

,CaseId,Activity,Timestamp
0,1,CREATE_ENTRY_SUCCESS,2025-01-01 09:00:00
1,2,DATA_INPUT_SUCCESS,2025-01-01 09:15:00
2,1,VALIDATE_DATA_SUCCESS,2025-01-01 09:25:00


In [19]:
xes.to_csv("xes.csv", index=False)
xes2.to_csv("xes2.csv", index=False)

# Alpha-Algorithmus (manuell)


 * create -> validateS -> decide -> close
 * input -> check
 * create -> validateF -> revise -> validateS -> decide -> close
 * create -> input -> check -> decide -> close
 * create -> input -> validateF -> revise -> validateF -> revise -> decide -> close


CREATE_ENTRY_SUCCESS, DATA_INPUT_SUCCESS, VALIDATE_DATA_SUCCESS, CHECK_RULES_SUCCESS, DECIDE_ACTION_SUCCESS, VALIDATE_DATA_FAILED, REVISE_ENTRY_SUCCESS, CLOSE_CASE_SUCCESS

## Matrix

|           | create | input | validateS | check | decide | validateF | revise | close |
| --------- | ------ | ----- | --------- | ----- | ------ | --------- | ------ | ----- |
| create    | #      | ->    | ->        | #     | #      | ->        | #      | #     |
| input     | <-     | #     | #         | ->    | #      | ->        | #      | #     |
| validateS | <-     | #     | #         | #     | ->     | #         | <-     | #     |
| check     | #      | <-    | #         | #     | ->     | #         | #      | #     |
| decide    | #      | #     | <-        | <-    | #      | #         | <-     | ->    |
| validateF | <-     | <-    | #         | #     | #      | #         | <->    | #     |
| revise    | #      | #     | ->        | #     | ->     | <->       | #      | #     |
| close     | #      | #     | #         | #     | <-     | #         | #      | #     |


* ({create}, {input})
* ({create}, {validateS})
* ({create}, {validateF})
* ({input}, {check})
* ({input}, {validateF})
* ({validateS}, {decide})
* ({check}, {close})
* ({decide}, {close})
* ({revise}, {validateS})
* ({revise}, {decide})

*Im Beispiel werden (scheinbar ohne muster) Aktivitäten als mengen zusammengefasst. Also mache ich das auch...
**Start:**
+ ({create}, {input, validateS, validateF}) -> p1
+ ({input}, {check, validateF}) -> p2

**??**
+ ({validateS}, {decide}) -> p3
+ ({revise}, {validateS, decide}) -> p6
  
+ ({create, revise}, {validateS}) -> p7
+ ({validateS, revised}, {decide}) -> p9
  
**End:**
+ ({input}, {check}) -> p10
+ ({check, decide}, {close}) -> p11

## Petri-Netz

![petri-netz.png](petri-netz.png)

# Alpha-Algorithmus